# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR$^2$ dataset using the `mlcroissant` library with explicit references to all fields, record sets, or columns by their `@id`s.

### Dataset Source
This dataset is defined by a Croissant schema accessible at:

[https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

_This exploration demonstrates how to:
- Load FAIR$^2$ Croissant metadata
- Inspect record sets and their field `@id`s
- Extract and process tabular data referencing fields by `@id`
- Perform basic exploratory data analysis (EDA) and visualization_


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Name: ' + metadata.name)
print('\nDescription: ' + metadata.description)

# For further exploration, we'll inspect the available record sets.

## 2. Data Overview
Let's review all available record sets, their fields, and relevant entity `@id`s in the dataset.

_The mlcroissant API exposes `dataset.record_sets` with entities that provide information on each record set structure._

In [ ]:
# List all record sets by their @id, name, and available field @id's

record_sets = list(dataset.record_sets)

if len(record_sets) == 0:
    print('No record sets available in this dataset.')
else:
    print(f"Found {len(record_sets)} record set(s):\n")
    for rs in record_sets:
        print(f'Record set @id: {rs["@id"]}')
        print(f'  Name: {rs.get("name", "[no name]")}' )
        print(f'  Fields:')
        for f in rs["fields"]:
            # Each field is available with a .metadata dictionary as well
            print(f'    - {f["@id"]} ({f.get("name", "[no name]")})')
        print()

## 3. Data Extraction
If the dataset exposes record sets, we can load them into DataFrames for analysis. The code below dynamically discovers all record sets and loads them using their `@id`.

_If there are zero record sets, this section will demonstrate empty example outputs. Otherwise, it shows how to extract data from each available record set._

In [ ]:
# Automatically extract data from all record sets and load as DataFrames
record_set_ids = [r["@id"] for r in dataset.record_sets]
dataframes = {}

if len(record_set_ids) == 0:
    print('No record sets available for extraction.')
else:
    for record_set_id in record_set_ids:
        # Each record yields a dictionary keyed by field @ids
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Record set {record_set_id} - shape {dataframes[record_set_id].shape}")
            print(f"Columns (field @ids): {dataframes[record_set_id].columns.tolist()}")
            display(dataframes[record_set_id].head())
        else:
            print(f"Record set {record_set_id} has no records.")

# For demonstration in this notebook, select the first record set, if any
if len(dataframes):
    demo_record_set_id = list(dataframes.keys())[0]
    print(f"Selected record set for further analysis: {demo_record_set_id}")
    demo_df = dataframes[demo_record_set_id]
else:
    demo_record_set_id = None
    demo_df = None

## 4. Exploratory Data Analysis (EDA)
Now let's perform some sample data processing using field `@id`s. We'll:
- Filter records based on a numeric field
- Normalize that field (z-score)
- Optionally group by a categorical field if available

All field references are done using their `@id`s.

_If the dataset does not have record sets, this cell will only print guidance text._

In [ ]:
# Demo: Only run if we have a DataFrame to work with
if demo_df is None or demo_df.shape[0] == 0:
    print('No data available for EDA. Please check if the dataset has record sets with data.')
else:
    # List numeric fields by checking dtypes (All columns are identified by their field @id)
    numeric_field_candidates = [c for c in demo_df.columns if pd.api.types.is_numeric_dtype(demo_df[c])]
    
    if len(numeric_field_candidates) == 0:
        print('No numeric fields found for EDA in the selected record set.')
    else:
        numeric_field_id = numeric_field_candidates[0]
        print(f'Using numeric field @id: {numeric_field_id}')
        # Choose a simple threshold: the mean
        threshold = demo_df[numeric_field_id].mean()
        filtered_df = demo_df[demo_df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Add normalized column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Try grouping by a likely categorical field (using first non-numeric if any)
        cat_field_candidates = [c for c in demo_df.columns if not pd.api.types.is_numeric_dtype(demo_df[c])]
        if len(cat_field_candidates):
            group_field_id = cat_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} by {group_field_id} (grouped):")
            display(grouped_df.head())
        else:
            print("No categorical field found to group by.")

## 5. Visualization
Finally, let's visualize data distributions or field relationships using `matplotlib` and `seaborn`. All axes and fields will be referenced by their `@id`s.

_If there are no record sets, no plots will be created._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if demo_df is None or demo_df.shape[0] == 0:
    print('No data available for visualization.')
else:
    # Plot histogram of the numeric field
    if len(numeric_field_candidates):
        nf = numeric_field_candidates[0]
        plt.figure(figsize=(7,4))
        sns.histplot(demo_df[nf], kde=True)
        plt.title(f'Distribution of field {nf}')
        plt.xlabel(nf)
        plt.show()

    # If a categorical field exists, make a boxplot
    if len(numeric_field_candidates) and len(cat_field_candidates):
        cf = cat_field_candidates[0]
        plt.figure(figsize=(10,4))
        sns.boxplot(x=cf, y=nf, data=demo_df)
        plt.title(f'{nf} by category {cf}')
        plt.xlabel(cf)
        plt.ylabel(nf)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load a structured dataset defined by a Croissant schema using `mlcroissant`
- Identify and reference record sets, fields, and columns by their `@id`
- Extract and process records for analysis in pandas
- Perform basic filtering, normalization, grouping, and visualization directly referencing schema-level field identities

This provides a reproducible and standards-based workflow for FAIR data exploration.
